In [16]:
import os
import ast
import glob
import pickle
import itertools
from collections import Counter
from functools import reduce

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

import econml
from econml.grf import CausalForest

import dask
import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar

import concurrent.futures
from multiprocessing import Pool, cpu_count
from concurrent.futures import ThreadPoolExecutor
from joblib import Parallel, delayed

import dask_ml.cluster as dask_cluster

from tqdm.notebook import tqdm

from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score

pd.set_option('display.max_columns', None)

# Uncomment the following line if needed
# from sklearn.cluster import KMeans

# Uncomment the following line if needed
# from tqdm import tqdm

from pprint import pprint


### Read Dataset

In [2]:
destfile = ("../../data/augmented_us-counties-states_latest_variants.csv")
augmented_panel_data = dd.read_csv(destfile, assume_missing=True)

hhs_X_w_clusters_fpath = "../benchmark_tcv_kmeans_code/hhs_X_w_clusters.csv"
hhs_X_w_clusters = dd.read_csv(hhs_X_w_clusters_fpath, assume_missing=True).compute()
hhs_X_w_clusters = hhs_X_w_clusters.sort_values(by="fips")

### Add Transformations

In [5]:
early_data = augmented_panel_data[augmented_panel_data["days_from_start"] <= 300]
early_data = early_data[early_data["rolled_cases"] >= 20]
early_data["log_rolled_cases"] = np.log(early_data["rolled_cases"] + 1)
early_data = early_data.compute()
early_data = early_data.sort_values(by=["fips", "days_from_start"])
early_data['shifted_days_from_start'] = early_data.groupby('fips')['days_from_start'].transform(lambda x: x - x.min())

NameError: name 'head' is not defined

In [6]:
early_data

,fips,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_NOHSDP,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_MOBILE,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_FIPS_Code,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Closed_day_cares,Reopen_day_cares,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,Stay_at_home_order'_issued_but_did_not_specifically_restrict_movement_of_the_general_public,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Closed_businesses_overnight,Began_to_reopen_businesses,Religious_gatherings_exempt_without_clear_social_distance_mandate*,Face_mask_mandate_in_public_spaces,Face_mask_mandate_x2,Face_mask_mandate_enforced_by_fines,Face_mask_mandate_enforced_by_criminal_charge/citation,No_legal_enforcement_of_face_mask_mandate,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Ended_face_mask_mandate_x2,Attempted_to_prevent_local_governments_from_implementing_face_mask_orders,Banned_local_mask_mandates,Liquor_stores_remained_open,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Firearm_sellers_remained_open,Cannabis_dispensaries_considered_essential_business,Closed_restaurants_except_take_out,Reopen_restaurants,Initially_reopen_restaurants_for_outdoor_dining_only,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Allowed_businesses_to_reopen_overnight,Began_to_reclose_bars,Closed_bars_(x2),Closed_movie_theaters_(x2),Closed_hair_salons/barber_shops_(x2),Closed_gyms_(x2),Closed_restaurants_(x2),Reopened_restaurants_(x2),Reopened_bars_(x2),Reopened_gyms_(x2),Reopened_hair_salons/barber_shops_(x2),Reopened_movie_theaters_(x2),Closed_bars_(x3),Closed_restaurants_(x3),Reopened_bars_(x3),Reopened_restaurants_(x3),Mandate_quarantine_for_those_entering_the_state_from_specific_settings,Mandate_quarantine_for_all_individuals_entering_the_state,Date_all_mandated_quarantines_ended,Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_People_who_are_Incarcerated,Vaccine_allocation_phase:_Residents_of_Homeless_Shelters,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers

In [7]:
case_number_columns = ["fips","date","county","state","days_from_start","log_rolled_cases", "shifted_days_from_start"]
early_data_case_numbers = early_data[case_number_columns]
X_time_invariant = hhs_X_w_clusters[list(set(early_data.columns).intersection(hhs_X_w_clusters.columns))]
X_time_invariant = X_time_invariant.drop(columns=["county","state"])

In [10]:
WYX = pd.merge(early_data_case_numbers, X_time_invariant, on="fips")
WYX

,fips,date,county,state,days_from_start,log_rolled_cases,shifted_days_from_start,EPL_MOBILE,F_DISABL,RPL_THEME3,Proof_of_residency_requirement_for_vaccination,Reinstated_work_search_requirement_for_UI,Different_Minimum_Wage_for_Smaller_Businesses,F_GROUPQ,Expanded_eligibility_of_unemployment_insurance_to_those_who_have_lost_childcare/school_closures,Were_there_casino(s)_in_State?_(Land_and_Non-Land_based),Waived_all_copays_during_pandemic_for_incarcerated_individuals,F_AGE17,MP_LIMENG,EP_NOHSDP,EP_MUNIT,E_CROWD,Vaccine_allocation_phase:_Healthcare_Service_Workers,RPL_THEME2,MP_MUNIT,Paid_sick_leave_prior_to_pandemic,F_CROWD,hhs_region,E_PCI,Adults_ages_65+_prioritized_ahead_of_essential_workers,F_POV,EP_MOBILE,Taxable_Wage_Amount,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Require_earnings_in_the_last_two_calendar_quarters_of_the_base_period_in_order_to_qualify_for_UI,Report_Indigenous_COVID-19_testing,Air_or_ventilation_standards,F_MINRTY,Waived_work_search_requirement_for_unemployment_insurance,Vaccine_allocation_phase:_Frontline_Essential_Workers,EPL_PCI,Vaccine_allocation_phase:_Correctional_Staff,Branch_of_government_implementing_COVID-19_liability_rules,EPL_MINRTY,Minimum_total_earnings_required_in_the_base_period_to_qualify_for_UI,MP_AGE17,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Proof_of_work_eligibility_requirement_for_vaccination,Proof_of_age_eligibility_requirement_for_vaccination,MP_UNEMP,Firearm_sellers_remained_open,EP_UNINSUR,EP_AGE17,Total_Unemployment_Rate_at_Extended_Benefits_program_shutoff,F_LIMENG,Closed_businesses_overnight,Mention_of_tribal_casinos,MP_AGE65,SPL_THEMES,Report_COVID-19_cases_by_race/ethnicity,Oct_1_2019_Minimum_Wage,COVID-19_anti-retaliation_rules,Number_Homeless_(2019),RPL_THEME1,Reinstated_one_week_waiting_period_for_unemployment_insurance,"Mental_health_professionals_per_100,000_population_in_2019",Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Adults_Ages_65+,Extended_the_amount_of_time_an_individual_can_be_on_unemployment_insurance,EPL_MUNIT,Vaccine_allocation_phase:_Long-term_Care_Residents,Aerosol_transmissable_diseases_standards,Report_COVID-19_vaccinations_by_race/ethnicity,Report_Indigenous_COVID-19_vaccinations,EP_UNEMP,COVID-19_liability_protections_for_healthcare_providers,Penalty_for_failure_to_comply_with_vaccine_distribution_requirements,F_TOTAL,Vaccine_allocation_phase:_EMS_Providers,Made_Effort_to_Limit_Abortion_Access,EP_GROUPQ,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Higher_Education_Employees,Last_date_of_receipt_of_mail-in_ballot_request_for_the_general_election_(by_mail_or_online),F_THEME1,Percent_at_risk_for_serious_illness_due_to_COVID,Cannabis_dispensaries_considered_essential_business,Percent_Unemployed_(2018),MP_CROWD,EPL_CROWD,MP_PCI,Suspend_CHIP_premium_non-payment_lock-outs,EP_SNGPNT,MP_MOBILE,SPL_THEME3,RPL_THEME4,Religious_gatherings_exempt_without_clear_social_distance_mandate*,COVID-19_business_liability_protections,E_UNEMP,SPL_THEME2,E_MUNIT,Total_Unemployment_Rate_relative_to_prior_year_(%),Vaccine_allocation_phase:_Home_Healthcare_Workers,Minimum_Tax_Rate_(%),OSHA-approved_state_plan,MP_GROUPQ,MP_UNINSUR,SPL_THEME4,Report_Indigenous_COVID-19_cases,F_SNGPNT,State_FIPS_Code,F_MOBILE,Vaccine_allocation_phase:_Grocery_Store_Workers,EPL_SNGPNT,F_THEME3,"Automatic_applications_sent_for_mail-in_ballots_in_response_to_COVID-19_(0,1,2_2_being_conditional-_see_notes_for_details)",Report_COVID-19_deaths_by_race/ethnicity,positiveScore,Vaccine_allocation_phase:_Additional_Essential_Workers,EP_AGE65,"Occupational_injuries,_illnesses,_and_fatalities_reporting_prior_to_pandemic",EP_DISABL,Face_mask_mandate_enforced_by_criminal_charge/citation,E_SNGPNT,Vaccine_allocation_phase:_Firefighters,Liquor_stores_remained_open,MP_MINRTY,Expanded_eligibility_to_high-risk_individuals_in_preventative_quarantine,Data_reporting_required_for_medical_treatment,Expanded_scope_of_practice_o

In [11]:
Y = WYX["log_rolled_cases"]
W = WYX["shifted_days_from_start"]

X = WYX[WYX.columns.difference(case_number_columns)]

In [12]:
X

,Adults_ages_65+_prioritized_ahead_of_essential_workers,Aerosol_transmissable_diseases_standards,Air_or_ventilation_standards,"Automatic_applications_sent_for_mail-in_ballots_in_response_to_COVID-19_(0,1,2_2_being_conditional-_see_notes_for_details)","Automatic_mail-in_ballot_system_in_response_to_COVID-19_(0,1,2_2_being_conditional-_see_notes_for_details)",Average_Benefit_Amount_(August),Branch_of_government_implementing_COVID-19_liability_rules,COVID-19_anti-retaliation_rules,COVID-19_business_liability_protections,COVID-19_liability_protections_for_healthcare_providers,COVID-19_workers'_compensation_expansion,Cannabis_dispensaries_considered_essential_business,Closed_businesses_overnight,Data_reporting_required_for_medical_treatment,Did_not_waive_copays_for_incarcerated_individuals,Different_Minimum_Wage_for_Smaller_Businesses,Does_not_charge_copays_for_incarcerated_individuals,EPL_AGE17,EPL_AGE65,EPL_CROWD,EPL_DISABL,EPL_GROUPQ,EPL_LIMENG,EPL_MINRTY,EPL_MOBILE,EPL_MUNIT,EPL_NOHSDP,EPL_NOVEH,EPL_PCI,EPL_POV,EPL_SNGPNT,EPL_UNEMP,EP_AGE17,EP_AGE65,EP_CROWD,EP_DISABL,EP_GROUPQ,EP_LIMENG,EP_MINRTY,EP_MOBILE,EP_MUNIT,EP_NOHSDP,EP_NOVEH,EP_PCI,EP_POV,EP_SNGPNT,EP_UNEMP,EP_UNINSUR,E_CROWD,E_GROUPQ,E_MUNIT,E_NOVEH,E_PCI,E_SNGPNT,E_UNEMP,E_UNINSUR,Expanded_eligibility_of_unemployment_insurance_to_anyone_who_is_quarantined_and/or_taking_care_of_someone_who_is_quarantined,Expanded_eligibility_of_unemployment_insurance_to_those_who_have_lost_childcare/school_closures,Expanded_eligibility_to_high-risk_individuals_in_preventative_quarantine,Expanded_scope_of_practice_of_certain_health_providers_to_administer_vaccines,Extended_the_amount_of_time_an_individual_can_be_on_unemployment_insurance,F_AGE17,F_AGE65,F_CROWD,F_DISABL,F_GROUPQ,F_LIMENG,F_MINRTY,F_MOBILE,F_MUNIT,F_NOHSDP,F_NOVEH,F_PCI,F_POV,F_SNGPNT,F_THEME1,F_THEME2,F_THEME3,F_THEME4,F_TOTAL,F_UNEMP,Face_mask_mandate_enforced_by_criminal_charge/citation,Face_mask_mandate_enforced_by_fines,Firearm_sellers_remained_open,Initially_reopen_restaurants_for_outdoor_dining_only,Jan_1_2019_Minimum_Wage,Jul_1_2019_Minimum_Wage,LAT,LON,Last_date_of_receipt_of_mail-in_ballot_request_for_the_general_election_(by_mail_or_online),Liquor_stores_remained_open,MP_AGE17,MP_AGE65,MP_CROWD,MP_DISABL,MP_GROUPQ,MP_LIMENG,MP_MINRTY,MP_MOBILE,MP_MUNIT,MP_NOHSDP,MP_NOVEH,MP_PCI,MP_POV,MP_SNGPNT,MP_UNEMP,MP_UNINSUR,Made_Effort_to_Limit_Abortion_Access,Mar_29_2019_Minimum_Wage,Maximum_Tax_Rate_(%),Medicaid_Expansion,"Mental_health_professionals_per_100,000_population_in_2019",Mention_of_tribal_casinos,Minimum_Tax_Rate_(%),Minimum_number_of_work_days_missed_by_workers_to_require_data_reporting,Minimum_total_earnings_required_in_the_base_period_to_qualify_for_UI,Minimum_total_earnings_required_outside_the_highest_earning_calendar_quarter_of_the_base_period_to_qualify_for_UI,No_legal_enforcement_of_face_mask_mandate,No_state_unemployment_waiting_period_prior_to_pandemic;_or_date_waiting_period_waived_not_found,Number_Homeless_(2019),Number_of_calendar_quarters_with_earnings_in_the_base_period_needed_to_qualify_for_UI,OSHA-approved_state_plan,"Occupational_injuries,_illnesses,_and_fatalities_reporting_prior_to_pandemic",Oct_1_2019_Minimum_Wage,Paid_sick_leave_expansion_during_pandemic,Paid_sick_leave_prior_to_pandemic,Penalty_for_failure_to_comply_with_vaccine_distribution_requirements,Percent_Unemployed_(2018),Percent_at_risk_for_serious_illness_due_to_COVID,Percent_living_under_the_federal_poverty_line_(2018),Permanent_mail-in_ballot_system,Population_density_per_square_miles,Prioritization_by_race/ethnicity,Proof_of_age_eligibility_requirement_for_vaccination,Proof_of_residency_requirement_for_vaccination,Proof_of_work_eligibility_requirement_for_vaccination,RPL_THEME1,RPL_THEME2,RPL_THEME3,RPL_THEME4,RPL_THEMES,Reinstated_one_week_waiting_period_for_unemployment_insurance,Reinstated_work_search_requirement_for_UI,Religious_gatherings_exempt_without_clear_social_distance_mandate*,Report_COVID-19_cases_by_race/e

In [17]:
grf = CausalForest()

In [35]:
grf.fit(X,W,Y)

ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

In [19]:
W

0           0.0
1           1.0
2           2.0
3           3.0
4           4.0
          ...  
445972    246.0
445973    247.0
445974    248.0
445975    249.0
445976    250.0
Name: shifted_days_from_start, Length: 445977, dtype: float64

In [32]:
Y.max()

11.582127647883352

In [33]:
Y.min()

2.995732273553991

In [34]:
W

0           0.0
1           1.0
2           2.0
3           3.0
4           4.0
          ...  
445972    246.0
445973    247.0
445974    248.0
445975    249.0
445976    250.0
Name: shifted_days_from_start, Length: 445977, dtype: float64